# PSS paths 

# Setup

## Library import
We import all the required Python libraries

The non-default libaries are networkX (https://networkx.org/) and py4cytoscape (https://py4cytoscape.readthedocs.io/, only necessary if you wish to view the results in Cytoscape). 

You will also need the skm-tools package. 

In [1]:
from pathlib import Path
from datetime import datetime
import networkx as nx

The skm-tools package: 

In [2]:
from skm_tools import load_networks, pss_utils

In [3]:
today = datetime.today().strftime('%Y.%m.%d'); today

'2026.06.30'

## Path and parameter definitions

In [5]:
base_dir = Path("./")
data_dir = base_dir / "data"
output_dir = base_dir / "output"

In [9]:
data_dir.mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)

## Load PSS

This code will download use the latest live PSS instance. 

For a description of the files, see [skm.nib.si/downloads](https://skm.nib.si/downloads). 

In [12]:
pss_edge_path = data_dir / f"rxn-edges-public-{today}.tsv"
pss_node_path = data_dir / f"rxn-nodes-public-{today}.tsv"

In [13]:
g = load_networks.pss_to_networkx(
    edge_path=pss_edge_path, 
    node_path=pss_node_path
)

print(f"\nNumber of nodes: {g.number_of_nodes()}\nNumber of edges: {g.number_of_edges()}")


Number of nodes: 752
Number of edges: 1583


# Filter

In [14]:
# Define the types we want to keep
keep_types = [
    'Complex',
    'Condition',
    'ForeignAbiotic',
    'Metabolite',
    'PlantAbstract',
    'PlantCoding',
    'PlantNonCoding',
    'Process'
]

# Define the species we want to keep
species = ['ath', 'stu']

In [15]:
# Use the util function to filter PSS
removed = pss_utils.filter_pss_nodes(g, 
                                     node_types=keep_types, 
                                     species=species, 
                                     remove_isolates=True
)

Removed 84 nodes from network.


In [16]:
print(f"\nNumber of nodes: {g.number_of_nodes()}\nNumber of edges: {g.number_of_edges()}")


Number of nodes: 668
Number of edges: 1384


# Path extraction

Studying crosstalk between JA nad SA.

First we identify the nodes of interest.

In [19]:
JA = [x for x,y in g.nodes(data=True) if y['name']=="JA"][0]
SA = [x for x,y in g.nodes(data=True) if y['name']=="SA"][0];

print(JA)
print(SA)

JA
SA


## JA --> SA

In [20]:
JA_paths = [p for p in nx.all_shortest_paths(g, source=JA, target=SA)]
JA_paths

[['JA',
  'JA-Ile',
  'COI1|JA-Ile|SCF',
  'JAZ[AT1G17380,AT1G19180,AT1G30135,AT1G48500,AT1G70700,AT1G72450,AT1G74950,AT2G34600,AT3G17860,AT3G43440,AT5G13220,AT5G20900,FUN_003441,FUN_004843,FUN_013768,FUN_016368,FUN_026609,FUN_032462,FUN_039026,FUN_039096,MALDO.HC.V1A1.CH10A.G02520,MALDO.HC.V1A1.CH10A.G02521,MALDO.HC.V1A1.CH13A.G09221,MALDO.HC.V1A1.CH13A.G10189,MALDO.HC.V1A1.CH14A.G14180,MALDO.HC.V1A1.CH15A.G16427,MALDO.HC.V1A1.CH15A.G16476,MALDO.HC.V1A1.CH15A.G18562,MALDO.HC.V1A1.CH16A.G18869,MALDO.HC.V1A1.CH16A.G19823,MALDO.HC.V1A1.CH17A.G22651,MALDO.HC.V1A1.CH17A.G23000,MALDO.HC.V1A1.CH2A.G27138,MALDO.HC.V1A1.CH2A.G27231,MALDO.HC.V1A1.CH5A.G37282,MALDO.HC.V1A1.CH5A.G37283,MALDO.HC.V1A1.CH6A.G40321,MALDO.HC.V1A1.CH9A.G47422,PAF106G0100002672,PAF106G0100003921,PAF106G0300014171,PAF106G0400017348,PAF106G0500021249,PAF106G0700026787,PAF106G0700026864,PCER_002028-RA,PCER_003058-RA,PCER_007287-RA,PCER_008287-RA,PCER_012621-RA,PCER_023697-RA,PCER_028370-RA,PCER_029970-RA,PCER_032863-RA,PCE

## SA --> JA + SA --> ROS

In [21]:
SA_paths = [p for p in nx.all_shortest_paths(g, source=SA, target=JA)]
SA_paths

[['SA',
  'CAT[AT1G20620,AT1G20630,AT4G35090,FUN_024095,MALDO.HC.V1A1.CH16A.G21388,MALDO.HC.V1A1.CH6A.G38166,MALDO.HC.V1A1.CH6A.G38170,PAF106G0500018582,PCER_026258-RA,PCER_026260-RA,PCER_037538-RA,PCER_037540-RA,PCER_083799-RA,PCER_083801-RA,PCER_086410-RA,PRAM_26145.1.P1,PRUARM.5G016000,PRUARM.5G016200,PRUPE.5G011300,PRUPE.5G011400,PYRCO.DA.V2A1.CHR16A.208880,PYRCO.DA.V2A1.CHR6A.424510,PYRCO.DA.V2A1.SNAP.424490,SOLTU.DM.02G022700,SOLTU.DM.04G037660,SOLTU.DM.12G004810,SOLYC04T002988,SOLYC04T002990,SOLYC12T002504,SOLYC12T002505,SOTUB12G027890.1.1,TEXASF1_G17381,TEXASF1_G17383,TEXASF1_G27937,TEXASF1_G27940,VITVI05_01CHR18G01320]',
  'ACX[AT1G06290,AT2G35690,AT4G16760,AT5G65110,FUN_015276,FUN_021007,FUN_024738,MALDO.HC.V1A1.CH13A.G10192,MALDO.HC.V1A1.CH15A.G17466,MALDO.HC.V1A1.CH17A.G23726,MALDO.HC.V1A1.CH1A.G24517,MALDO.HC.V1A1.CH4A.G32687,MALDO.HC.V1A1.CH9A.G48088,MALDO.HC.V1A1.CH9A.G48089,PAF106G0300013125,PAF106G0500019244,PAF106G0600023734,PCER_017645-RA,PCER_021164-RA,PCER_026815-R

# Cytoscape 

First open the Cytoscape application. Then the following cell will load the required library and and make sure you can connect to the Cytoscape application. 

More py4cytoscape documentation is here: https://py4cytoscape.readthedocs.io/

In [14]:
from skm_tools import cytoscape_utils

In [15]:
import py4cytoscape as p4c
p4c.cytoscape_ping()
p4c.cytoscape_version_info()

You are connected to Cytoscape!


{'apiVersion': 'v1',
 'cytoscapeVersion': '3.10.2',
 'automationAPIVersion': '1.9.0',
 'py4cytoscapeVersion': '1.9.0'}

We set the Cytoscape collection name for this notebook. 

In [16]:
COLLECTION = f"PSS: JA, SA, ROS ({today})"
COLLECTION

'PSS: JA, SA, ROS (2024.08.21)'

We're going to highlight the identified paths in Cytoscape, and we set the colours here:

In [17]:
JA_COLOUR = "#66a61e"
SA_COLOUR = "#34858d"
ROS_COLOUR = "#dc1c1c"

## Load the PSS network into Cytoscape

We load the network, set a visual style, and apply the CoSE layout.

With skm-tools, we provide a default style for PSS, colouring the nodes bypathway.

Returned is the ID of the network view in Cytoscape.

In [18]:
pss_network_suid = p4c.networks.create_network_from_networkx(g, title="Complete PSS", collection=COLLECTION)
cytoscape_utils.apply_builtin_style(pss_network_suid, 'pss')
p4c.layout_network("cose", network=pss_network_suid)
pss_network_suid

Applying default style...
Applying preferred layout
Applied PSS-default to 53457


53457

Now we're going to highlight the paths we identified in the network by applying style bypasses.

We don't want to recolour already highlighted path elements, so we keep track of them here:

In [19]:
done_nodes, done_edges = [], []

In [20]:
for p in JA_paths:
    done_nodes_now, done_edges_now = cytoscape_utils.highlight_path(p, JA_COLOUR, skip_nodes=done_nodes, skip_edges=done_edges)
    done_nodes += done_nodes_now
    done_edges += done_edges_now

JA JAR[AT2G46370,AT4G03400]
JAR[AT2G46370,AT4G03400] JA-Ile
JA-Ile COI1|JA-Ile|SCF
COI1|JA-Ile|SCF JAZ[AT1G17380,AT1G19180,AT1G30135,AT1G48500,AT1G70700,AT1G72450,AT1G74950,AT2G34600,AT3G17860,AT3G43440,AT5G13220,AT5G20900]
JAZ[AT1G17380,AT1G19180,AT1G30135,AT1G48500,AT1G70700,AT1G72450,AT1G74950,AT2G34600,AT3G17860,AT3G43440,AT5G13220,AT5G20900] EIN3(like)[AT2G27050,AT3G20770]
EIN3(like)[AT2G27050,AT3G20770] ICS[AT1G18870,AT1G74710]
ICS[AT1G18870,AT1G74710] IsoChor
IsoChor PBS3[AT5G13320]
PBS3[AT5G13320] IsoChor-9-Glu
IsoChor-9-Glu EPS1[AT5G67160]
EPS1[AT5G67160] SA
JA JAR[AT2G46370,AT4G03400]
JAR[AT2G46370,AT4G03400] JA-Ile
JA-Ile COI1|JA-Ile|SCF
COI1|JA-Ile|SCF JAZ[AT1G17380,AT1G19180,AT1G30135,AT1G48500,AT1G70700,AT1G72450,AT1G74950,AT2G34600,AT3G17860,AT3G43440,AT5G13220,AT5G20900]
JAZ[AT1G17380,AT1G19180,AT1G30135,AT1G48500,AT1G70700,AT1G72450,AT1G74950,AT2G34600,AT3G17860,AT3G43440,AT5G13220,AT5G20900] MYC2[AT1G32640]
MYC2[AT1G32640] ICS[AT1G18870,AT1G74710]
ICS[AT1G18870,AT1G74

In [21]:
for p in SA_paths:
    print(p)
    done_nodes_now, done_edges_now = cytoscape_utils.highlight_path(p, SA_COLOUR, skip_nodes=done_nodes, skip_edges=done_edges)
    done_nodes += done_nodes_now
    done_edges += done_edges_now


['SA', 'CAT[AT1G20620,AT1G20630,AT4G35090,SOTUB12G027890.1.1]', 'ACX[AT1G06290,AT2G35690,AT4G16760,AT5G65110,SOTUB10G008540.1.1]', 'OPC6-CoA', 'MFP[AT3G06860,AT3G15290,AT4G29010]', 'OPC4-CoA', 'KAT[AT1G04710,AT2G33150,AT5G48880]', 'JA-CoA', 'ACH[AT2G30720,AT5G48370]', 'JA']
SA CAT[AT1G20620,AT1G20630,AT4G35090,SOTUB12G027890.1.1]
CAT[AT1G20620,AT1G20630,AT4G35090,SOTUB12G027890.1.1] ACX[AT1G06290,AT2G35690,AT4G16760,AT5G65110,SOTUB10G008540.1.1]
ACX[AT1G06290,AT2G35690,AT4G16760,AT5G65110,SOTUB10G008540.1.1] OPC6-CoA
OPC6-CoA MFP[AT3G06860,AT3G15290,AT4G29010]
MFP[AT3G06860,AT3G15290,AT4G29010] OPC4-CoA
OPC4-CoA KAT[AT1G04710,AT2G33150,AT5G48880]
KAT[AT1G04710,AT2G33150,AT5G48880] JA-CoA
JA-CoA ACH[AT2G30720,AT5G48370]
ACH[AT2G30720,AT5G48370] JA
['SA', 'CAT[AT1G20620,AT1G20630,AT4G35090,SOTUB12G027890.1.1]', 'ROS']
No more nodes to colour


At this point, the Cytoscape session has a network view of the filtered PSS, and highlighting of the paths we extracted from our targeted searches. 

## Extract subnetworks in Cytoscape

Properly inspecting the identified paths is a bit hard within the complete network, so here we pull out the subnetworks of and surrounding the paths. 


### The edge induced network
The first, and smallest subnetwork, is created by extracting only the edges that are present on the paths. 

In [23]:
network_edge_induced_suid = cytoscape_utils.subnetwork_edge_induced_from_paths(
    paths=JA_paths + SA_paths + ROS_paths,
    g=g,
    parent_suid=pss_network_suid,
    name="identified paths (edge induced)",
)

We apply a new layout to this subnetwork

In [24]:
_ = p4c.layouts.layout_network('cose', network=network_edge_induced_suid)

### The node induced network

Now we extract the network based on the nodes along the paths, meaning any edges between those nodes that are not on the paths are also extracted. 

In [25]:
nodes = list(set([y for x in JA_paths + SA_paths + ROS_paths for y in x]))

In [26]:
network_node_induced_suid = cytoscape_utils.subnetwork_node_induced(
    nodes=nodes,
    parent_suid=pss_network_suid,
    name="identified paths (node induced)",
)

Instead of applying a network layout algorithm, we can copy the layout from the previous subnetwork. 

In [27]:
_ = p4c.layouts.layout_copycat(
    network_edge_induced_suid, 
    network_node_induced_suid
)

### Neighbours

For more context around our paths, we can include the first neighbours in the view. We can use the Cytoscape first neighbour selection functionality. 

In [28]:
network_neighbours_suid = cytoscape_utils.subnetwork_neighbours(
    nodes=nodes,
    parent_suid=pss_network_suid,
    name="identified paths + 1st neighbours",
)

In [29]:
_ = p4c.layouts.layout_network('cose', network=network_neighbours_suid)

This network (with manual layout improvements and removal of unrelated nodes) is shown in Figure 3 of the article. 

### Additional filtering of the neighbours

There are many neighbours displayed now, and we are perhaps only interested in the ones that are connected to at least two of the original path nodes, so we can make a filter using networkX neighbour functions. 

In [30]:
filtered_neighbours = []
for n in g.nodes():
    if (len([x for x in nx.MultiGraph(g).neighbors(n) if (x in done_nodes)]) > 1) and (n not in done_nodes):
        filtered_neighbours.append(n)

In [31]:
network_neighbours_filtered_suid = cytoscape_utils.subnetwork_node_induced(
    nodes=nodes+filtered_neighbours,
    parent_suid=pss_network_suid,
    name="identified paths + 1st neighbours (filtered)",
)

In [32]:
p4c.layouts.layout_copycat(
    network_neighbours_suid, 
    network_neighbours_filtered_suid
)

{'mappedNodeCount': 39, 'unmappedNodeCount': 0}

Save the Cytoscape session:

## Saving and exporting

In [33]:
p4c.session.save_session(str(output_dir / f"PSS-JA-SA-ROS-{today}.cys"))

{}

In [34]:
collection_suid = p4c.get_collection_suid(network_edge_induced_suid)

In [35]:
from skm_tools import cytoscape_pdf_utils

In [36]:
cytoscape_pdf_utils.export_collection_to_pdfs(collection_suid, output_dir / "figures")

53457 Complete PSS
59991 identified paths (edge induced)
60348 identified paths (node induced)
60671 identified paths + 1st neighbours
61917 identified paths + 1st neighbours (filtered)
Collection saved to output/figures


In [37]:
cytoscape_pdf_utils.export_collection_to_single_pdf(collection_suid, output_dir / "figures" / "single_pdf", caption=True)

53457 Complete PSS
59991 identified paths (edge induced)
60348 identified paths (node induced)
60671 identified paths + 1st neighbours
61917 identified paths + 1st neighbours (filtered)
Collection save to output/figures/single_pdf


In [38]:
# END